In [21]:
%pip install nltk spacy plotly pandas nbformat

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 5.1 MB/s eta 0:00:00 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.2/33.2 MB 4.5 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 3.8 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 3.5 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.8/260.8 kB 3.3 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.1/134.1 kB 3.8 MB/s eta 0:00:00MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━

In [ ]:
!python -m spacy download en_core_web_sm

In [2]:
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

True

## Natural Language Processing (NLP) Test Environment

NLP will be used to extract information about password hash algorithms and salt status from the descriptions provided by HIBP. This notebook is for testing and comparing different approaches

In [10]:
import json
with open("testset.json", "r") as fp:
    test_set = json.load(fp)
algorithms = {}
allResults = {}

## Notes from creating the test set

The test set consists of a number of samples from the full API response data set as of February 2026, which have been hand-labelled. Samples are taken from various points in the data which is ordered by date added to ensure a variety of writing styles are present. Currently the number of samples is set to 100, which represents just over 10% of the data. The set was created using [generatetestset.py](generatetestset.py).

Labels were applied using the following rules:
1. isHashed:
  - isHashed = True iff the description states that the breach contained password hashes and there is no indication of plain-text passwords
  - isHashed = False iff the description states that the breach did not contain password hashes or did contain plain-text passwords
  - else isHashed = None (i.e. the description makes no clear indication either way)
2. isSalted:
  - if isHashed != True then isSalted = None
  - isSalted = True iff the description states that the password hashes in the breach were salted
  - isSalted = False iff the description states that the password hashes in the breach were not salted 
  - else isSalted = None (i.e. the description makes no clear indication either way)
  - if the description indicates the presence of both salted and unsalted hashes, isSalted = False (i.e. take the weaker form)
3. hashAlgo
  - if isHashed != True then hashAlgo = None
  - if the description states hashes were in a specific form (e.g. MD5 or SHA-1), then set hashAlgo to the string name of that hash algorithm
  - if the description mentions two or more algorithms were present, take the weaker one (see [here](#hash-functions))
  - else (if unknown) hashAlgo = None

As in any data set, there are edge cases that need to be handled carefully. In the following example, the description mentions that passwords were present in plain text, but made it clear that this was the result of cracking and the original data contained salted MD5 hashes:

> "In November 2016, the game developer <a href="https://www.hackread.com/vbulletin-forums-hacked-accounts-sold-on-dark-web/" target="_blank" rel="noopener">Suba Games suffered a data breach</a> which led to the exposure of 6.1M unique email addresses. Impacted data also included usernames and passwords, most of which appeared circulating in the breached file in plain text after being cracked from salted MD5 hashes. The data was provided to HIBP by <a href="https://dehashed.com/" target="_blank" rel="noopener">dehashed.com</a>."

Normally, if the text states passwords were present in two or more forms, we would take the weaker form, however since this text makes it clear that plain text was not the original form, we can mark it as being salted MD5 hashes instead.

Another case of potential ambiguity is the following example:

> In January 2023, <a href="https://www.digi.no/artikler/selger-datalekkasje-med-140-000-berorte-kunder-fra-lars-monsens-nettbutikk/525070" target="_blank" rel="noopener">the online Norwegian store KomplettFritid was reported as having had a data breach dating back to February 2021</a>. The incident exposed 140k customer records including physical, email and IP addresses, names, phone numbers and passwords. Most passwords were stored as bcrypt hashes with a small number appearing in plain text.

The text mentions that the majority of passwords were bcrypt hashes, with only a small number of plain text passwords. While it seems unfair to mark it as plain text, the labelling rules we set out mean this has to be done for consistency.


## Hash Functions

Below is a list of known hash functions, ordered by increasing strength. This ordering is determined by the following sources in order:

- https://cheatsheetseries.owasp.org/cheatsheets/Password_Storage_Cheat_Sheet.html
- https://csrc.nist.gov/projects/hash-functions
- https://web.archive.org/web/20090521001714/http://www.infosec.sdu.edu.cn/uploadfile/papers/How%20to%20Break%20MD5%20and%20Other%20Hash%20Functions.pdf

Any other hash functions should be considered equally weak. It should also be noted that at the top end of the scale, the strength differences between each algorithm are minimal and a service's choice is dependent on their context, however an order is still necessary for our purposes.

1. MD5, MD4, SHA-0, RipeMD
2. SHA-1
3. SHA-2 family (SHA-256, SHA-512, etc.)
9. SHA3 family (SHA3-256, SHA3-512, etc.)
13. bcrypt
14. PBKDF2
15. scrypt
16. Argon2

In [11]:
HASH_STRENGTH_ORDER = [
    "RIPEMD", "MD4","MD5",
    "SHA-1",
    "SHA-256", "SHA-512",
    "SHA3-256", "SHA3-512",
    "bcrypt",
    "PBKDF2",
    "scrypt",
    "Argon2",
]

HASH_FUNCTION_ALIASES = {
    "SHA1": "SHA-1",
    "SHA256": "SHA-256",
    "SHA512": "SHA-512",
    "sha1": "sha-1",
    "sha256": "sha-256",
    "sha512": "sha-512"
}

### Algorithm 1: Basic String Matching

In [3]:
def basicStringMatching(text:str):
    text = text.lower()
    # is hashed?
    isHashed = None
    if any([x in text for x in ("hashed","hashes")]):
        isHashed = True
    # Check for negation
    if any([x in text for x in ("not hashed","unhashed","plain text")]):
        isHashed = False

    # is salted
    isSalted = None
    if any([x in text for x in ("salted", "salt")]):
        isSalted = True
    # Check for negation
    if any([x in text for x in ("unsalted", "not salted", "no salt")]):
        isSalted = False

    # Find the hash algorithm used
    hashAlgo = None
    # Check for a number of known algorithms
    # Check in order of hash strength so if multiple hashes used, we report the weakest used
    for a in HASH_STRENGTH_ORDER:
        if a.lower() in text:
            hashAlgo = a
            break
    
    # also check for aliases
    for k, v in HASH_FUNCTION_ALIASES.items():
        if k.lower() in text:
            hashAlgo = v
            break  

    return (isHashed, isSalted, hashAlgo)

algorithms["Basic String Matching"] = basicStringMatching

### Algorithm 2: String Matching + Dependency Matching

String matching performs very well for isHashed and isSalted, but not for hashAlgo, so use spaCy Dependency Matching for this.

In [4]:
# Testing
import spacy,re
from spacy import displacy

nlp = spacy.load("en_core_web_sm")
t = "Salted scrypt password hashes for users who <em>didn't</em> sign up with either Google or Facebook authentication were also included.".lower()
t = re.sub('<.*?>', '', t)
text = nlp(t)
displacy.render(text, style="dep")

In [5]:
import spacy
from spacy.matcher import DependencyMatcher

nlp = spacy.load("en_core_web_sm")
matcher = DependencyMatcher(nlp.vocab)

pattern = [
    {
        "RIGHT_ID":"anchor_hashes",
        "RIGHT_ATTRS": {"ORTH": {"in": ["hashes", "passwords"]}}
    },
    {
        "LEFT_ID": "anchor_hashes",
        "REL_OP": ">",
        "RIGHT_ID": "hash_compound",
        "RIGHT_ATTRS": {"DEP": "compound"},
    }
]
matcher.add("HASH_ALGO", [pattern])

def stringAndDependencyMatching(text:str):
    text = text.lower()
    # is hashed?
    isHashed = None
    if any([x in text for x in ("hashed","hashes")]):
        isHashed = True
    # Check for negation
    if any([x in text for x in ("not hashed","unhashed","plain text")]):
        isHashed = False

    # is salted
    isSalted = None
    if any([x in text for x in ("salted", "salt")]):
        isSalted = True
    # Check for negation
    if any([x in text for x in ("unsalted", "not salted", "no salt")]):
        isSalted = False

    # Find the hash algorithm used
    foundAlgos = set()
    doc = nlp(text)
    matches = matcher(doc)
    # print("\n".join([doc[token_id].text for _, token_ids in matches for token_id in token_ids]))
    # take first match
    for match in matches:
        match_id, token_ids = match
        a = doc[token_ids[1]].text
        if a in ("password", "user", "text"): continue
        if a in HASH_FUNCTION_ALIASES.keys(): a = HASH_FUNCTION_ALIASES[a]
        foundAlgos.add(a) # 1 is index of hash_compound in the pattern

    hashAlgo = None
    for hashFunc in HASH_STRENGTH_ORDER:
        if hashFunc.lower() in foundAlgos:
            hashAlgo = hashFunc
            break

    return (isHashed, isSalted, hashAlgo)

algorithms["String + Dependency Matching"] = stringAndDependencyMatching
# basicStringMatching("In June 2018, online fashion retailer <a href=\"https://www.zdnet.com/article/shein-fashion-retailer-announces-breach-affecting-6-42-million-users/\" target=\"_blank\" rel=\"noopener\">SHEIN suffered a data breach</a>. The company discovered the breach 2 months later in August then disclosed the incident another month after that. A total of 39 million unique email addresses were found in the breach alongside MD5 password hashes.")
stringAndDependencyMatching("Salted SHA-1 password hashes for users who <em>didn't</em> sign up with either Google or Facebook authentication were also included.")

(True, True, None)

### Algorithm 3: String Matching + Information Extraction

Alternative to algorithm 2 using NLTK Information Extraction

In [6]:
# Testing
import nltk, pprint, re
t = "Salted sha-512 password hashes for users who <em>didn't</em> sign up with either Google or Facebook authentication were also included.".lower()
t = re.sub('<.*?>', '', t)
t = re.sub('[-.,!?]', '', t)
toks = nltk.word_tokenize(t)
pos = nltk.pos_tag(toks)
grammar = "NP: {<VBN>?<NN>+<NNS>}"
rp = nltk.RegexpParser(grammar)
parsed = rp.parse(pos)
print(parsed)

(S
  (NP salted/VBN sha512/NN password/NN hashes/NNS)
  for/IN
  users/NNS
  who/WP
  did/VBD
  n't/RB
  sign/VB
  up/RP
  with/IN
  either/DT
  google/NN
  or/CC
  facebook/NN
  authentication/NN
  were/VBD
  also/RB
  included/VBN)


In [7]:
import nltk, re

def stringAndInformationExtraction(text:str):
    text = text.lower()
    # is hashed?
    isHashed = None
    if any([x in text for x in ("hashed","hashes")]):
        isHashed = True
    # Check for negation
    if any([x in text for x in ("not hashed","unhashed","plain text")]):
        isHashed = False

    # is salted
    isSalted = None
    if any([x in text for x in ("salted", "salt")]):
        isSalted = True
    # Check for negation
    if any([x in text for x in ("unsalted", "not salted", "no salt")]):
        isSalted = False

    # Find the hash algorithm used
    t = re.sub('<.*?>', '', text)
    t = re.sub('[-.,!?]', '', t)
    tokens = nltk.word_tokenize(t)
    pos = nltk.pos_tag(tokens)
    
    # perform noun-phrase chunking
    grammar = r"NP: {<VBN>?<NN>+<NNS>}"
    rp = nltk.RegexpParser(grammar)
    parsed = rp.parse(pos)

    hashAlgo = None
    stemmer = nltk.PorterStemmer()
    for node in parsed:
        try:
            node.label() # type: ignore # duck typing; if it walks like a duck...
        except AttributeError:
            continue
        else:
            if node.label() == "NP":  # type: ignore
                leaves = node.leaves() # type: ignore
                stemmed = [stemmer.stem(leaf[0]) for leaf in leaves]
                if "password" in stemmed or "hash" in stemmed:
                    hashAlgo = leaves[0][0]
                    if hashAlgo in HASH_FUNCTION_ALIASES: 
                        hashAlgo = HASH_FUNCTION_ALIASES[hashAlgo]
                    break

    return (isHashed, isSalted, hashAlgo)

algorithms["String Matching + Information Extraction"] = stringAndInformationExtraction
basicStringMatching("In June 2018, online fashion retailer <a href=\"https://www.zdnet.com/article/shein-fashion-retailer-announces-breach-affecting-6-42-million-users/\" target=\"_blank\" rel=\"noopener\">SHEIN suffered a data breach</a>. The company discovered the breach 2 months later in August then disclosed the incident another month after that. A total of 39 million unique email addresses were found in the breach alongside MD5 password hashes.")
# stringAndInformationExtraction("Salted SHA1 password hashes for users who <em>didn't</em> sign up with either Google or Facebook authentication were also included.")

(True, None, 'MD5')

### Algorithm 4: Ask an LLM

LLM technology has shown itself to be incredibly useful in NLP workloads, though has some inherent issues due to non-deterministic output and high resource consumption - overkill for this application but worth seeing how it performs.

In [8]:
# %pip install ollama pydantic

In [13]:
def llmParsing(text:str):
    import ollama, re
    from pydantic import BaseModel
    class LLMResponse(BaseModel):
        # reasoning: str
        is_hashed: bool
        certain_about_is_hashed: bool
        is_salted: bool
        certain_about_is_salted: bool
        hash_function: str
        certain_about_hash_function: bool
        # reasoning:str
    t = re.sub('<.*?>', '', text)
    response = ollama.chat(
        # model="gpt-oss:20b",
        model="gemma3:4b",
        messages=[{
            "role": "user", 
            "content": f"""In the following description of a data breach, identify if passwords in the breach were hashed, if those hashes were salted, and which hash function was used. Respond using JSON. Also indicate if you are certain about your answers. 
            \"{t}\""""
        }],
        format=LLMResponse.model_json_schema(),
        options={"temperature": 0},
        stream=False
        )
    print(response.message.content)
    llm_response = LLMResponse.model_validate_json(response.message.content) # type: ignore

    # De-alias hash function names
    hash_func = llm_response.hash_function if llm_response.certain_about_hash_function else None
    if hash_func in HASH_FUNCTION_ALIASES: hash_func = HASH_FUNCTION_ALIASES[hash_func]

    return llm_response.is_hashed if llm_response.certain_about_is_hashed else None, llm_response.is_salted if llm_response.certain_about_is_salted else None, hash_func

algorithms["LLM (Gemma3:4b)"] = llmParsing
# llmParsing("In June 2018, online fashion retailer <a href=\"https://www.zdnet.com/article/shein-fashion-retailer-announces-breach-affecting-6-42-million-users/\" target=\"_blank\" rel=\"noopener\">SHEIN suffered a data breach</a>. The company discovered the breach 2 months later in August then disclosed the incident another month after that. A total of 39 million unique email addresses were found in the breach alongside MD5 password hashes.")

## Evaluation

In [14]:
import random
def dummyAlgo(_):
    return (random.choice((True, False, None)), random.choice((True, False, None)), random.choice(("MD5", "SHA-1", None)))
    # return (None, None, None)

# algorithms["dummy"] = dummyAlgo

In [15]:
import re

def assignEvalMatrix(truth, predicted, matrix):
    truthI = 0 if truth == True else 1 if truth == False else 2
    predictedI = 0 if predicted == True else 1 if predicted == False else 2
    matrix[predictedI][truthI] += 1

hashAlgos = []
# algorithms must be defined
for i, algo in enumerate(algorithms):
    print("--------------------")
    print(f"{algo}:")

    # Get test outputs first
    results = []
    for test in test_set:
        text = test['description']
        text = re.sub('<.*?>', '', text)
        # isHashed, isSalted, hashAlgo = algorithms[algo](test)
        results.append(algorithms[algo](text))

    hashAlgos.append(sorted(set([test["algorithm"].lower() if test["algorithm"] is not None else None for test in test_set]+[result[2].lower() if result[2] is not None else None
                       for result in results]), key=lambda x: "" if x == None else x))
    print(hashAlgos[i])
    n = len(hashAlgos[i])

    isHashedEvalMatrix = [[0 for _ in range(3)] for _ in range(3)]
    isSaltedEvalMatrix = [[0 for _ in range(3)] for _ in range(3)]
    hashAlgoEvalMatrix = [[0 for _ in range(n)] for _ in range(n)]

    for j, test in enumerate(test_set):
        isHashed, isSalted, hashAlgo = results[j]
        # print(test)
        assignEvalMatrix(test['isHashed'], isHashed, isHashedEvalMatrix)
        assignEvalMatrix(test['isSalted'], isSalted, isSaltedEvalMatrix)

        hashAlgo = hashAlgo.lower() if hashAlgo is not None else None
        truthI = hashAlgos[i].index(test['algorithm'].lower() if test['algorithm'] is not None else None)
        predI = hashAlgos[i].index(hashAlgo)
        hashAlgoEvalMatrix[predI][truthI] += 1

    allResults[algo] = (isHashedEvalMatrix, isSaltedEvalMatrix, hashAlgoEvalMatrix)
    print(allResults[algo])
    

--------------------
LLM (Gemma3:4b):
{"is_hashed": true,
 "certain_about_is_hashed": true,
 "is_salted": false,
 "certain_about_is_salted": true,
 "hash_function": "MD5",
 "certain_about_hash_function": true}

{"is_hashed": true,
 "certain_about_is_hashed": true,
 "is_salted": true,
 "certain_about_is_salted": false,
 "hash_function": "Unknown",
 "certain_about_hash_function": false}
 
{"is_hashed": false,
 "certain_about_is_hashed": true,
 "is_salted": false,
 "certain_about_is_salted": true,
 "hash_function": "Unknown",
 "certain_about_hash_function": false
}

{"is_hashed": true,
 "certain_about_is_hashed": true,
 "is_salted": true,
 "certain_about_is_salted": false,
 "hash_function": "Unknown",
 "certain_about_hash_function": false
}

{"is_hashed": true,
 "certain_about_is_hashed": true,
 "is_salted": true,
 "certain_about_is_salted": false,
 "hash_function": "Unknown",
 "certain_about_hash_function": false
}

{"is_hashed": true,
 "certain_about_is_hashed": true,
 "is_salted": true

In [16]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=len(allResults), cols=3,
    # shared_xaxes=True,
    vertical_spacing=0.1,
    horizontal_spacing=0.01,
    specs=[[{"type": "table"}]*3 for _ in range(len(allResults))],
    subplot_titles=[f"{algo}: {t}" for algo in allResults for t in ["isHashed","isSalted","hashAlgorith "]],
    column_widths=[2,2,3]
)

for i, algo in enumerate(allResults):
    isHashedEvalMatrix, isSaltedEvalMatrix, hashAlgoEvalMatrix = allResults[algo]
    # Add tables to plot
    # fig = go.Figure(data=[
    fig.add_trace(
        go.Table(
            header=dict(
                values=[["Truth\\Pred"],["True"],["False"],["Unknown"]],
                # font=dict(size=10),
                align="center",
                fill_color="lightgrey"
            ),
            cells=dict(
                values=[["True","False","Unknown"]]+isHashedEvalMatrix,
                align="center",
                fill=dict(color=["lightgrey", "white"])
            ),
        ),
        row=i+1,
        col=1
        # ],
    )

    fig.add_trace(
        go.Table(
            header=dict(
                values=[["Truth\\Pred"],["True"],["False"],["Unknown"]],
                # font=dict(size=10),
                align="center",
                fill_color="lightgrey"
            ),
            cells=dict(
                values=[["True","False","Unknown"]]+isSaltedEvalMatrix,
                align="center",
                fill=dict(color=["lightgrey", "white"])
            ),
        ),
        row=i+1,
        col=2
        # ],
    )

    fig.add_trace(
        go.Table(
            header=dict(
                values=[["Truth\\Pred"]]+[[hashAlgos[i][j]] for j in range(len(hashAlgoEvalMatrix))],
                # font=dict(size=10),
                align="center",
                fill_color="lightgrey",

            ),
            cells=dict(
                values=[[hashAlgos[i][j] for j in range(len(hashAlgoEvalMatrix))]]+hashAlgoEvalMatrix,
                align="center",
                fill=dict(color=["lightgrey", "white"])
            ),
        ),
        row=i+1,
        col=3
        # ],
    )

fig.update_layout(autosize=True, margin=dict(l=20, r=20, t=30, b=20), height=600)

fig.show()

In [17]:
for algo in allResults:
    print(f"{algo}:")
    for a, property in enumerate(("isHashed","isSalted","hashAlgo")):
        matrix = allResults[algo][a]
        k = len(matrix)
        tp, tn, fp, fn = [0]*k, [0]*k, [0]*k, [0]*k
        for i in range(k):
            # TP_i = n_ii
            tp[i] = matrix[i][i]
            # FN_i = sum_{j!=i} sum_{k!=i} n_jh
            tn[i] = sum([r for j, c in enumerate(matrix) for h, r in enumerate(c) if j!=i and h!=i])
            # FN_i = sum_{j!=i} n_ij
            fn[i] = sum([c[i] for j, c in enumerate(matrix) if j!=i])
            # FP_i = sum_{j!=i} n_ji
            fp[i] = sum([r for j, r in enumerate(matrix[i]) if j!=i])
            # print(f"TP_i = {tp[i]}, TN_i = {tn[i]}, FP_i = {fp[i]}, FN_i = {fn[i]}")

        # Micro = sum TP, TN etc first then calculate acc. prec. etc.
        # Macro = calc acc. prec. for each class then average

        TP = sum(tp)
        TN = sum(tn)
        FP = sum(fp)
        FN = sum(fn)
        # print(f"{property} Macro-averages:")
        # print(f"TP = {TP/k:.03}, TN = {TN/k:.03}, FP = {FP/k:.03}, FN = {FN/k:.03}")
        # print(f"{property} Micro-averages:")
        # print(f"TP = {TP}, TN = {TN}, FP = {FP}, FN = {FN}")

        pre = TP / (TP + FP)
        rec = TP / (TP + FN)
        spe = TN / (TN + FP)
        print(f"{property} Micro Accuracy: {(TP+TN) / (TP+TN+FP+FN) :0.2}")
        print(f"{property} Micro Precision: {(TP) / (TP+FP) :0.2}")
        print(f"{property} Micro F1: {(2*pre*rec) / (pre+rec) :0.2}")
        print(f"{property} Micro precision, recal, specificity: {pre:0.2} {rec:0.2} {spe:0.2}")

        # print(f"{property} Macro Accuracy: {sum([(tp[i] + tn[i])/(tp[i]+tn[i]+fp[i]+fn[i]) for i in range(k)])/k:0.2}")
        # print(f"{property} Macro Precision: {sum([(tp[i])/(tp[i]+fp[i]) if (tp[i]+fp[i])!=0 else 1 for i in range(k)])/k:0.2}")
        # print(f"{property} Macro F1: {sum([(2*tp[i])/(2*tp[i]+fp[i]+fn[i]) if (tp[i]+fp[i]+fn[i])!=0 else 1 for i in range(k)])/k:0.2}")

LLM (Gemma3:4b):
isHashed Micro Accuracy: 0.88
isHashed Micro Precision: 0.82
isHashed Micro F1: 0.82
isHashed Micro precision, recal, specificity: 0.82 0.82 0.91
isSalted Micro Accuracy: 0.79
isSalted Micro Precision: 0.69
isSalted Micro F1: 0.69
isSalted Micro precision, recal, specificity: 0.69 0.69 0.84
hashAlgo Micro Accuracy: 0.99
hashAlgo Micro Precision: 0.94
hashAlgo Micro F1: 0.94
hashAlgo Micro precision, recal, specificity: 0.94 0.94 0.99


## Notes

### LLM Performance

```text
--------------------
LLM (Gemma3:4b):
[None, 'argon2', 'bcrypt', 'bcrypt and sha-512', 'md5', 'md5 and bcrypt', 'mybb', 'pbkdf2', 'scrypt', 'sha-1', 'sha-512', 'sha1', 'unknown']
([[71, 4, 0], [2, 13, 10], [0, 0, 0]], [[31, 0, 25], [0, 10, 23], [0, 0, 11]], [[24, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 13, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], [3, 0, 0, 0, 31, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0], [8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
```

```text
LLM (Gemma3:4b):
TP_i = 71, TN_i = 23, FP_i = 4, FN_i = 2
TP_i = 13, TN_i = 71, FP_i = 12, FN_i = 4
TP_i = 0, TN_i = 90, FP_i = 0, FN_i = 10
isHashed Micro Accuracy: 0.89
isHashed Micro Precision: 0.84
isHashed Micro F1: 0.84
isHashed Macro Accuracy: 0.89
isHashed Macro Precision: 0.82
isHashed Macro F1: 0.53
TP_i = 31, TN_i = 44, FP_i = 25, FN_i = 0
TP_i = 10, TN_i = 67, FP_i = 23, FN_i = 0
TP_i = 11, TN_i = 41, FP_i = 0, FN_i = 48
isSalted Micro Accuracy: 0.68
isSalted Micro Precision: 0.52
isSalted Micro F1: 0.52
isSalted Macro Accuracy: 0.68
isSalted Macro Precision: 0.62
isSalted Macro F1: 0.5
TP_i = 24, TN_i = 64, FP_i = 0, FN_i = 12
TP_i = 1, TN_i = 99, FP_i = 0, FN_i = 0
TP_i = 13, TN_i = 86, FP_i = 1, FN_i = 0
TP_i = 0, TN_i = 99, FP_i = 1, FN_i = 0
TP_i = 31, TN_i = 65, FP_i = 3, FN_i = 1
TP_i = 0, TN_i = 99, FP_i = 1, FN_i = 0
TP_i = 1, TN_i = 99, FP_i = 0, FN_i = 0
TP_i = 5, TN_i = 95, FP_i = 0, FN_i = 0
TP_i = 1, TN_i = 99, FP_i = 0, FN_i = 0
TP_i = 6, TN_i = 92, FP_i = 0, FN_i = 2
TP_i = 2, TN_i = 97, FP_i = 0, FN_i = 1
TP_i = 0, TN_i = 98, FP_i = 2, FN_i = 0
TP_i = 0, TN_i = 92, FP_i = 8, FN_i = 0
hashAlgo Micro Accuracy: 0.98
hashAlgo Micro Precision: 0.84
hashAlgo Micro F1: 0.84
hashAlgo Macro Accuracy: 0.98
hashAlgo Macro Precision: 0.68
hashAlgo Macro F1: 0.64
```